# 技能1 · Day 1 上机：营销文本表示学习

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **sentence-transformers** 将营销评论编码为 384 维 embedding
2. 用 **scikit-learn** 做 t-SNE/PCA 降维可视化和 KMeans 聚类，理解表示空间的几何结构
3. 用 **torch** 实现 Autoencoder 压缩表示，理解"压缩-重建"瓶颈
4. 用 **DSR 六步框架**定义"企业表示工程"研究问题，把工程实践转化为学术贡献

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：sentence-transformers（UKPLab/sentence-transformers，18.9k★）+ scikit-learn + torch。
营销映射：20条产品评论（护肤/电子/健身 × 正面/负面），用 embedding 编码后做降维/压缩/聚类/分类。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> ⚠️ sentence-transformers 首次运行会自动下载 all-MiniLM-L6-v2 模型（约 90MB），需网络。
> 模型缓存到 ~/.cache/huggingface/，后续运行无需网络。
> 如需更好的中文支持，可将模型名替换为 `paraphrase-multilingual-MiniLM-L12-v2`（同为 384 维）。

In [ ]:
# !pip install sentence-transformers scikit-learn torch -q

## 1. 数据集背景与营销映射

**处理对象**：20 条真实营销场景的产品评论（护肤/电子/健身三类 × 正面/负面两种情感）。

| 类别 | 正面 | 负面 | 示例 |
|------|------|------|------|
| 护肤 | 4条 | 4条 | "这款烟酰胺精华液真的太好用了，用了两周肤色明显提亮..." |
| 电子 | 3条 | 3条 | "跑步手表功能很全面，GPS轨迹精准，续航14天不用充..." |
| 健身 | 3条 | 3条 | "瑜伽垫材质很好，防滑效果一流，做下犬式再也不滑了..." |

每条评论包含：
- `review`：评论文本（中文，50-100字）
- `category`：产品类别（skincare / electronics / fitness）
- `sentiment`：情感标签（positive / negative）

**营销映射**：在真实项目中，这些数据来自电商平台的用户评价系统。表示工程的目标是把文本评论转化为可计算的向量，用于客户分群、情感分析、推荐系统。

**理论连接**：从 `f(x)=wᵀφ(x)`（手工特征）到 `f(x)=wᵀφ_θ(x)`（端到端学习）的范式转移--sentence-transformers 的 `φ_θ` 是预训练的，直接将文本映射为语义向量。

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np

# sentence-transformers: 将文本编码为 embedding
from sentence_transformers import SentenceTransformer

# scikit-learn: 降维、聚类、评估
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# torch: 自编码器
import torch
import torch.nn as nn

print("依赖导入完成")

## 1：用 sentence-transformers 编码营销评论

In [ ]:
# 1. 用 sentence-transformers 编码营销评论为 embedding
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(reviews, show_progress_bar=True)

print(f"Embedding 维度: {embeddings.shape}")
print(f"单条评论向量示例（前10维）: {embeddings[0][:10].round(4)}")

## 2. 表示学习理论基础回顾

### 范式转移：从手工特征到端到端学习

```
传统范式：f(x) = wᵀ φ(x)       -- φ 人工设计、固定
端到端范式：f(x) = wᵀ φ_θ(x)   -- φ_θ 数据驱动学习（θ 是可训练参数）
```

在营销场景中：传统方法需手动设计"浏览次数""购物车金额"等特征；端到端学习从原始行为文本中自动学习"比较后收藏但未购买 = 等待降价意图"这类序列模式。

### CMU 10741 三个核心概念

1. **不加约束的表示学习没有意义**：不限制维度，模型会退化为 lookup table（记忆而非学习）。约束（如 384 维）迫使模型发现潜在结构。
2. **Neural Collapse**（Papyan et al., 2020）：分类网络训练后期，最后一层特征呈现类别内方差趋零、类别间距离最大化的几何结构。好的表示让"相似的聚在一起，不同的分开"。
3. **不可辨识性**：不同随机种子训练的 embedding 数值不同但语义等价。不能解释单个维度，应关注几何关系（余弦相似度）。

### 非线性降维：为什么要用 t-SNE

t-SNE 用 t 分布（而非高斯分布）度量低维空间相似度，t 分布有更重的尾巴，解决了"中等距离的点降维后全挤一起"的拥挤问题。营销应用：把客户 embedding 投影到 2D，观察群体结构。

## 2-3：降维可视化 + 自编码器压缩

In [ ]:
# 2. 用 t-SNE 和 PCA 降维可视化
pca = PCA(n_components=2, random_state=42)
embeddings_pca = pca.fit_transform(embeddings)

tsne = TSNE(n_components=2, perplexity=5, random_state=42, n_iter=1000)
embeddings_tsne = tsne.fit_transform(embeddings)

print("PCA 降维结果：")
for i in range(len(reviews)):
    print(f"  评论{i+1:2d} [{sentiments[i]:8s}] PCA=({embeddings_pca[i,0]:.2f}, {embeddings_pca[i,1]:.2f})")

print("\nt-SNE 降维结果：")
for i in range(len(reviews)):
    print(f"  评论{i+1:2d} [{sentiments[i]:8s}] tSNE=({embeddings_tsne[i,0]:.2f}, {embeddings_tsne[i,1]:.2f})")

print(f"\nPCA 解释方差比: {pca.explained_variance_ratio_.round(4)}")
print(f"PCA 累计解释方差: {pca.explained_variance_ratio_.sum():.4f}")

In [ ]:
# 3. 用 torch 实现 Autoencoder 压缩 embedding（384 -> 64）
class Autoencoder(nn.Module):
    def __init__(self, input_dim=384, latent_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon, z

# 转换为 torch tensor
X = torch.FloatTensor(embeddings)

# 初始化模型
autoencoder = Autoencoder(input_dim=384, latent_dim=64)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)

# 训练 200 轮
for epoch in range(200):
    optimizer.zero_grad()
    recon, z = autoencoder(X)
    loss = criterion(recon, X)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d}, 重构损失: {loss.item():.6f}")

# 获取压缩表示
with torch.no_grad():
    _, compressed = autoencoder(X)

print(f"压缩表示维度: {compressed.shape}")
print(f"最终重构损失: {loss.item():.6f}")

## 3. 聚类与表示质量

### KMeans + Silhouette 评估

KMeans 在 embedding 空间中分群，Silhouette Score 衡量"类内紧凑、类间分离"的程度：

```
s(i) = (b(i) - a(i)) / max(a(i), b(i))
```

其中 `a(i)` 是样本到同簇其他点的平均距离，`b(i)` 是样本到最近其他簇的平均距离。s 越接近 1 表示聚类越好。

### 表示质量评估的两种方式

1. **无监督（内部指标）**：Silhouette Score --不需要标签，直接看向量空间的聚类结构
2. **有监督（下游任务）**：用 embedding 做情感分类，准确率高 = 表示质量好

这两种方式互补：无监督指标看"表示空间的几何结构"，有监督指标看"表示对下游任务的有效性"。

## 4：KMeans 聚类发现评论分群

In [ ]:
# 4. 用 KMeans 聚类发现评论分群，silhouette 评估最优 K
best_k = 2
best_score = -1

print("不同 K 值的 Silhouette Score：")
for k in range(2, 6):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings)
    score = silhouette_score(embeddings, labels)
    print(f"  K={k}, Silhouette={score:.4f}")
    if score > best_score:
        best_score = score
        best_k = k

# 最终聚类
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)

print(f"\n最优 K 值: {best_k}, Silhouette Score: {best_score:.4f}")
print("\n聚类结果 vs 情感标签：")
for i in range(len(reviews)):
    print(f"  评论{i+1:2d} [真实:{sentiments[i]:8s}] -> 聚类{cluster_labels[i]}")

# 分析聚类与情感/类别的对应关系
print("\n聚类 vs 情感交叉表：")
for cluster_id in range(best_k):
    members = [i for i in range(len(reviews)) if cluster_labels[i] == cluster_id]
    pos = sum(1 for i in members if sentiments[i] == 'positive')
    neg = sum(1 for i in members if sentiments[i] == 'negative')
    cats = [categories[i] for i in members]
    print(f"  聚类{cluster_id}: {len(members)}条 (正面:{pos}, 负面:{neg})")

## 4. DSR 六步框架：从工程实践到学术贡献

设计科学研究（Design Science Research）是信息系统领域的核心研究范式（Hevner 2004 / Peffers 2007），通过设计和评估 artifact 产生新知识。

**六步流程**：
1. 问题识别与动机 -- 从"标签"到"向量"的 gap
2. 定义解决方案目标 -- 统一表示框架
3. 设计与开发 -- embedding 系统 + Two-Tower + 对比学习
4. 演示 -- 在真实营销场景中验证
5. 评估 -- Recall@K / Silhouette / 跨域匹配准确率
6. 传播 -- IMRaD 论文投稿

**关键思考**：DSR 让工程实践有学术贡献框架--不是"做了系统"，而是"设计了新框架并验证有效性"。这正是博士级研究和硕士级项目的本质区别。

## 5：用 DSR 六步框架定义研究问题

In [ ]:
# 5. 用 DSR 六步框架定义"企业表示工程"研究问题
# 基于 Hevner et al. (2004) 和 Peffers et al. (2007) 的设计科学研究方法论
dsr_framework = {
    "Step1_问题识别与动机": (
        "企业营销数据目前用'标签'表示（年龄/性别/城市/消费等级），标签是人设计的、"
        "静态的、粗粒度的。AI时代需要用'向量'表示，向量是数据学习的、动态的、细粒度的。"
        "这个从'标签'到'向量'的 gap 就是研究问题。"
    ),
    "Step2_定义解决方案目标": (
        "设计一个企业营销数据的统一表示框架，能同时处理客户、产品、内容三类对象，"
        "支持跨域对齐（客户向量与产品向量可直接计算相似度）。"
    ),
    "Step3_设计与开发": (
        "构建客户/产品/内容的 embedding 系统：用 sentence-transformers 编码文本，"
        "用 Two-Tower 架构实现跨域对齐，用对比学习优化表示空间。"
        "Artifact = 统一表示框架（模型 + 方法 + 评估流程）。"
    ),
    "Step4_演示": (
        "在真实营销场景中展示：用 20 条产品评论验证 embedding 的聚类质量和分类准确率，"
        "在电商推荐场景中验证客户-产品匹配效果。"
    ),
    "Step5_评估": (
        "下游任务评估：1) 推荐准确率 Recall@K  2) 分群质量 Silhouette Score  "
        "3) 跨域匹配准确率 4) 与传统标签方法的 A/B 对比。"
    ),
    "Step6_传播": (
        "写一篇符合 IMRaD 格式的论文，投稿信息系统或 AI 营销相关期刊/会议。"
        "产出可复用的设计原则：'在中文营销文本场景中，多语言 embedding 模型比单语言模型"
        "的 Silhouette Score 高 X%'。"
    ),
}

for step, content in dsr_framework.items():
    print(f"{step}:")
    print(f"  {content}\n")

## 5. 表示质量评估

好的表示应该让下游任务表现好。我们用两种方式评估：

1. **下游分类准确率**：用 embedding 做情感分类（positive/negative），用 5 折交叉验证计算准确率
2. **Silhouette 对比**：比较原始 384 维表示和 Autoencoder 压缩后的 64 维表示的 Silhouette Score

**期望结果**：
- 原始 384 维表示的分类准确率应高于随机（50%）
- 压缩到 64 维后，准确率可能略降但不应大幅下降--说明 Autoencoder 保留了核心信息
- Silhouette Score 的变化反映压缩对表示空间几何结构的影响

## 6：评估表示质量（下游分类 + Silhouette 对比）

In [ ]:
# 6. 评估表示质量（下游分类准确率 + Silhouette 对比）
y = np.array([1 if s == 'positive' else 0 for s in sentiments])

# 原始 384 维表示：情感分类
clf = LogisticRegression(max_iter=1000, random_state=42)
scores_orig = cross_val_score(clf, embeddings, y, cv=5, scoring='accuracy')

# 压缩 64 维表示：情感分类
compressed_np = compressed.numpy()
scores_comp = cross_val_score(clf, compressed_np, y, cv=5, scoring='accuracy')

# Silhouette 对比（用情感标签作为真实标签）
sil_orig = silhouette_score(embeddings, y)
sil_comp = silhouette_score(compressed_np, y)

print("=" * 55)
print(f"原始 384 维表示：分类准确率 = {scores_orig.mean():.4f} +/- {scores_orig.std():.4f}")
print(f"压缩  64 维表示：分类准确率 = {scores_comp.mean():.4f} +/- {scores_comp.std():.4f}")
print(f"原始 384 维 Silhouette = {sil_orig:.4f}")
print(f"压缩  64 维 Silhouette = {sil_comp:.4f}")
print("=" * 55)
print()
print("解读：")
print("  - 分类准确率 > 50% 说明 embedding 包含情感语义信息")
print("  - 压缩后准确率下降幅度小说明 Autoencoder 保留了核心信息")
print("  - Silhouette > 0 说明正负面评论在表示空间中有分离趋势")
print("  - 如准确率接近 50%，说明 all-MiniLM-L6-v2 对中文情感理解有限，")
print("    可尝试 paraphrase-multilingual-MiniLM-L12-v2（同为 384 维，支持中文）")

## 6. 反思与前沿

### 反思问题
1. 你的营销评论 embedding 在 t-SNE 降维后呈现什么聚类结构？正负面评论是否清晰分离？
2. Autoencoder 压缩到 64 维后，重构损失是多少？信息损失多大？
3. KMeans 聚类的最优 K 是多少？聚类结果与产品类别/情感标签的对应关系如何？
4. 原始 384 维和压缩 64 维表示在下游分类准确率上差异多大？说明什么？

### 2026 前沿：Representation Engineering（RepE）
Representation Engineering（Zou et al., 2023, arXiv 2310.01405）是 MIT/Center for AI Safety 提出的 AI 透明性方法，通过操控神经网络内部的高层表示来监测和干预模型行为：
- **表示监测**：读取模型内部表示，判断是否在"想"虚构信息--比只看输出更早发现幻觉
- **表示操控**：调整内部表示方向，引导模型行为（增强品牌调性、抑制夸大宣传）
- **与本 Day 的连接**：RepE 的理论基础正是表示学习--只有理解了 embedding 空间的几何结构，才能理解如何读取和操纵表示

> ⚠️ RepE 对应因果阶梯 L1（关联分析），不能替代 L2（A/B 测试）。定位为"开发期透明性工具"。

参考 [arXiv 2310.01405](https://arxiv.org/abs/2310.01405)（Representation Engineering）+ [Neural Collapse](https://arxiv.org/abs/2008.08186)（Papyan et al., 2020）。